#### Supplemental Figure 5 A, B, C, D 

In [ ]:
from paths import DATA_DIR, fig_dir
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import beta as beta_dist
import os
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch

In [ ]:
# User-adjustable parameters
save_fig = False
fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)
    
import matplotlib as mpl

mpl.rcParams.update({
    "font.size": 14,
})
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["path.simplify"] = False

In [ ]:
rng_seed = 42
ps = [0.8, 0.5, 0.2, 0.5, 0.8, 0.2]
n_trials = 7

prior_a = 1
prior_b = 1

x_min = 0.0
x_max = 1.0
x_points = 1200

figsize = (6.5, 8)
W, H = figsize  # from the first figure

figsize_2x2 = (
    W * 1.8,     # widen for 2 columns + gutter
    H / 3)       # preserve per-distribution height
    
line_color = "black"
line_width = 2

dot_color = "black"
dot_size = 5

arrow_color = "black"
arrow_width = 1.5
arrow_y_offset_frac = 0.07

despine_offset = 10

dpi = 300

#### SF 5 A

In [ ]:
# Setup
rng = np.random.default_rng(rng_seed)
x = np.linspace(x_min, x_max, x_points)

sns.set_style("white")

fig, axes = plt.subplots(
    nrows=6,
    ncols=1,
    sharex=True,
    figsize=figsize,
    constrained_layout=True,
)

# Plot each beta distribution
for i, (ax, p) in enumerate(zip(axes, ps), start=1):
    # Simulate weighted coin flips
    flips = rng.random(n_trials) < p
    successes = int(flips.sum())
    failures = int(n_trials - successes)

    # Posterior beta parameters
    a = prior_a + successes
    b = prior_b + failures

    # Beta pdf, mean, variance
    pdf = beta_dist.pdf(x, a, b)
    mean = a / (a + b)
    var = (a * b) / ((a + b) ** 2 * (a + b + 1))
    sd = np.sqrt(var)

    # Plot pdf
    ax.plot(x, pdf, color=line_color, lw=line_width)

    # Arrow representing variance (±1 SD in x), placed above the curve
    pdf_max = pdf.max() if np.isfinite(pdf).all() else 1.0
    y_arrow = pdf_max * (1.0 + arrow_y_offset_frac)

    x0 = max(x_min, mean - sd)
    x1 = min(x_max, mean + sd)

    ax.annotate(
        "",
        xy=(x1, y_arrow),
        xytext=(x0, y_arrow),
        arrowprops=dict(
            arrowstyle="<->",
            color=arrow_color,
            lw=arrow_width,
        ),
        zorder=6,
        annotation_clip=False,
    )

    # Dot at the mean, centered on the SD arrow height
    ax.plot(mean, y_arrow, marker="o", color=dot_color, ms=dot_size, zorder=7)

    # Ensure arrow and dot are always visible
    ax.set_ylim(-0.1, y_arrow * 1.10)

    # Titles and y labels
    ax.set_title(
        f"",
        color="k",
        pad=6,
    )
    ax.set_ylabel(
        f"Port {i}\nBeta({successes}, {failures})",
        rotation=0,
        labelpad=30,
        va="center",
        color="k",
    )

    # Axis styling
    ax.set_yticks([])
    ax.tick_params(axis="y", length=0)

    sns.despine(
        ax=ax,
        left=True,
        right=True,
        top=True,
        bottom=ax is not axes[-1],
        offset=despine_offset,
    )

    if ax is not axes[-1]:
        ax.tick_params(axis="x", labelbottom=False)
    if ax is axes[0]:
        ax.legend(
            handles=[
                Line2D([], [], marker="o", linestyle="None",
                       color=dot_color, markersize=dot_size, label=r"$\mu$"),
                Line2D([], [], marker=r"$\leftrightarrow$", linestyle="None",
                       color=arrow_color, markersize=14, label="+/- "+r"$\sigma$"),
            ],
            frameon=False,
            loc="upper right",bbox_to_anchor=(1.2,0.8)
        )
        # save port 1 success/failures
        s1, f1 = successes, failures

# Shared x-axis formatting
axes[-1].set_xlim(x_min, x_max)
axes[-1].set_xticks(np.linspace(x_min, x_max, 6))
axes[-1].tick_params(axis="x", colors="k", length=4, width=1)

# Save or show
if save_fig:
    fig.savefig(fig_path+'6port_beta_distributions', dpi=dpi, bbox_inches="tight", format="pdf")

plt.show()

#### SF 5 B

In [ ]:
# 2x2 figure reusing port 1 results from the prior figure: requires s1, f1 already set
a_base, b_base = prior_a + s1,     prior_b + f1
a_succ, b_succ = prior_a + s1 + 1, prior_b + f1
a_fail, b_fail = prior_a + s1,     prior_b + f1 + 1

panels = [
    ("Port 1", a_base, b_base),        # top-left
    ("+1 success", a_succ, b_succ),  # top-right
    ("Port 1", a_base, b_base),        # bottom-left
    ("+1 failure", a_fail, b_fail),  # bottom-right
]

fig2, axes2 = plt.subplots(
    2, 2, sharex=True,
    figsize=figsize_2x2,
    gridspec_kw=dict(wspace=0.55, hspace=0.25),
    constrained_layout=False,
)

for ax, (label, a, b) in zip(axes2.ravel(), panels):
    pdf = beta_dist.pdf(x, a, b)
    mean = a / (a + b)
    var = (a * b) / ((a + b) ** 2 * (a + b + 1))
    sd = np.sqrt(var)

    ax.plot(x, pdf, color=line_color, lw=line_width)

    pdf_max = pdf.max() if np.isfinite(pdf).all() else 1.0
    y_arrow = pdf_max * (1.0 + arrow_y_offset_frac)

    x0 = max(x_min, mean - sd)
    x1 = min(x_max, mean + sd)

    ax.annotate(
        "",
        xy=(x1, y_arrow),
        xytext=(x0, y_arrow),
        arrowprops=dict(arrowstyle="<->", color=arrow_color, lw=arrow_width),
        zorder=6,
        annotation_clip=False,
    )
    ax.plot(mean, y_arrow, marker="o", color=dot_color, ms=dot_size, zorder=7)
    ax.set_ylim(-0.1, y_arrow * 1.10)
    ax.set_ylabel(
        f"{label}\nBeta({a-1:.0f}, {b-1:.0f})",
        rotation=0,
        labelpad=30,
        va="center",
        color="k",
    )

    ax.set_yticks([])
    ax.tick_params(axis="y", length=0)

for ax in axes2[0, :]:
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
for ax in axes2[1, :]:
    ax.tick_params(axis="x", bottom=True, labelbottom=True, colors="k", length=4, width=1)

for ax in axes2.ravel():
    sns.despine(
        ax=ax,
        left=True,
        right=True,
        top=True,
        bottom=(ax in axes2[0, :]),
        offset=despine_offset,
    )

for ax in axes2[1, :]:
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(np.linspace(x_min, x_max, 6))

# Save or show
if save_fig:
    fig2.savefig(fig_path+'port1_update_beta_distributions', dpi=dpi, bbox_inches="tight", format="pdf")

plt.show()

#### SF 5 C

In [ ]:
# 2x2 figure reusing port 1 results from the prior figure: requires s1, f1 already set
a_base, b_base = prior_a + s1,     prior_b + f1
a_succ, b_succ = prior_a + s1 + 1, prior_b + f1
a_fail, b_fail = prior_a + s1,     prior_b + f1 + 1

panels = [
    ("Port 1", a_base, b_base),       # top-left
    ("+1 success", a_succ, b_succ),   # top-right
    ("Port 1", a_base, b_base),       # bottom-left
    ("+1 failure", a_fail, b_fail),   # bottom-right
]

fig2, axes2 = plt.subplots(
    2, 2, sharex=True,
    figsize=figsize_2x2,
    gridspec_kw=dict(wspace=0.55, hspace=0.25),
    constrained_layout=False,
)

# Base stats for overlays in the right column
pdf_base = beta_dist.pdf(x, a_base, b_base)
mean_base = a_base / (a_base + b_base)
var_base = (a_base * b_base) / ((a_base + b_base) ** 2 * (a_base + b_base + 1))
sd_base = np.sqrt(var_base)
y_base_at_mean = float(beta_dist.pdf(mean_base, a_base, b_base))
x0_base = max(x_min, mean_base - sd_base)
x1_base = min(x_max, mean_base + sd_base)

for ax, (label, a, b) in zip(axes2.ravel(), panels):
    pdf = beta_dist.pdf(x, a, b)
    mean = a / (a + b)
    var = (a * b) / ((a + b) ** 2 * (a + b + 1))
    sd = np.sqrt(var)

    ax.plot(x, pdf, color=line_color, lw=line_width)

#     # Overlays for right column only: base distribution (dotted),
#     # base mean (open circle), base variance (dotted arrow)
    if ax in axes2[:, 1]:
        ax.plot(x, pdf_base, color="k", lw=1.0, ls=":")

        # Mirror the mean/variance overlay BELOW the dotted curve
        y_base_overlay = max(
            0.0,
            y_base_at_mean - arrow_y_offset_frac * pdf_base.max()
        )

        ax.plot(
            mean_base, y_base_overlay,
            marker="o",
            ms=dot_size,
            markerfacecolor="none",
            markeredgecolor="k",
            markeredgewidth=1.2,
            zorder=7,
        )

        ax.annotate(
            "",
            xy=(x1_base, y_base_overlay),
            xytext=(x0_base, y_base_overlay),
            arrowprops=dict(
                arrowstyle="<->",
                color="k",
                lw=1.0,
                linestyle=":",
            ),
            zorder=6,
            annotation_clip=False,
        )

    pdf_max = pdf.max() if np.isfinite(pdf).all() else 1.0
    y_arrow = pdf_max * (1.0 + arrow_y_offset_frac)

    x0 = max(x_min, mean - sd)
    x1 = min(x_max, mean + sd)

    ax.annotate(
        "",
        xy=(x1, y_arrow),
        xytext=(x0, y_arrow),
        arrowprops=dict(arrowstyle="<->", color=arrow_color, lw=arrow_width),
        zorder=6,
        annotation_clip=False,
    )
    ax.plot(mean, y_arrow, marker="o", color=dot_color, ms=dot_size, zorder=8)
    ax.set_ylim(-0.1, y_arrow * 1.10)

    ax.set_ylabel(
        f"{label}\nBeta({a-1:.0f}, {b-1:.0f})",
        rotation=0,
        labelpad=30,
        va="center",
        color="k",
    )

    ax.set_yticks([])
    ax.tick_params(axis="y", length=0)

for ax in axes2[0, :]:
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
for ax in axes2[1, :]:
    ax.tick_params(axis="x", bottom=True, labelbottom=True, colors="k", length=4, width=1)

for ax in axes2.ravel():
    sns.despine(
        ax=ax,
        left=True,
        right=True,
        top=True,
        bottom=(ax in axes2[0, :]),
        offset=despine_offset,
    )

for ax in axes2[1, :]:
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(np.linspace(x_min, x_max, 6))

# Save or show
if save_fig:
    fig2.savefig(fig_path+'port1_update_beta_distr_with_dotted_base_distribution', dpi=dpi, bbox_inches="tight", format="pdf")

plt.show()

#### SF 5 D

In [ ]:
x = np.linspace(-4, 4, 400)

fig, ax = plt.subplots(figsize=(4, 2.5))

z1, z2, z3 = x, np.zeros_like(x), -x
e1, e2, e3 = np.exp(z1), np.exp(z2), np.exp(z3)
p1 = e1 / (e1 + e2 + e3)

ax.plot(x, p1, color="k", lw=2)

ax.spines["left"].set_position(("data", 0))
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

ax.set_yticks([])
ax.set_xticks([0])
ax.set_xticklabels(["0"])

# Save or show
if save_fig:
    fig.savefig(fig_path+'little_softmax', dpi=dpi, bbox_inches="tight", format="pdf")

plt.show()